<a href="https://colab.research.google.com/github/Godstouch/GNN-Student-Risk-Prediction-/blob/main/Synthetic_graph.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer


df = pd.read_csv("/content/Featured_Synthetic_school_data_with_proxy.csv")

print("Dataset shape:", df.shape)



y = df["dropout_risk"].values

print("\nClass distribution:")
print(pd.Series(y).value_counts().sort_index())


drop_columns = [
    "Student ID",
    "risk_score",
    "dropout_risk"
]

X_df = df.drop(columns=drop_columns)


categorical_features = X_df.select_dtypes(
    include=["object"]
).columns.tolist()

numerical_features = X_df.select_dtypes(
    exclude=["object"]
).columns.tolist()

print("\nNumerical features:", len(numerical_features))
print("Categorical features:", len(categorical_features))


preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numerical_features
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_features
        )
    ]
)

X = preprocessor.fit_transform(X_df)

print("\nFinal feature matrix shape:", X.shape)

Dataset shape: (2988, 46)

Class distribution:
0     747
1    1494
2     747
Name: count, dtype: int64

Numerical features: 28
Categorical features: 15

Final feature matrix shape: (2988, 174)


In [ ]:

# CONSTRUCT kNN GRAPH

from sklearn.neighbors import NearestNeighbors

K = 8

knn = NearestNeighbors(
    n_neighbors=K + 1,
    metric="euclidean"
)

knn.fit(X)

distances, indices = knn.kneighbors(X)


edge_list = []

for i in range(len(X)):

    # indices[i][0] is the student itself
    neighbors = indices[i][1:]

    for j in neighbors:
        edge_list.append([i, j])

edge_index = np.array(edge_list).T

print("\nGraph constructed!")
print("Number of nodes:", X.shape[0])
print("Number of edges:", edge_index.shape[1])
print("Edge index shape:", edge_index.shape)


Graph constructed!
Number of nodes: 2988
Number of edges: 23904
Edge index shape: (2, 23904)


In [ ]:
import sys
!{sys.executable} -m pip install torch_geometric

from torch_geometric.utils import to_undirected

import torch

edge_index = torch.tensor(
    edge_index,
    dtype=torch.long
)

edge_index = to_undirected(edge_index)

print("Undirected edge index shape:", edge_index.shape)
print("Number of edges:", edge_index.shape[1])

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 13.8 MB/s eta 0:00:00
Undirected edge index shape: torch.Size([2, 37478])
Number of edges: 37478


In [ ]:
from torch_geometric.data import Data

x_tensor = torch.tensor(
    X,
    dtype=torch.float
)

y_tensor = torch.tensor(
    y,
    dtype=torch.long
)

data = Data(
    x=x_tensor,
    edge_index=edge_index,
    y=y_tensor
)

print(data)

Data(x=[2988, 174], edge_index=[2, 37478], y=[2988])


In [ ]:
from sklearn.model_selection import train_test_split

num_nodes = len(y)

all_indices = np.arange(num_nodes)

train_idx, temp_idx = train_test_split(
    all_indices,
    test_size=0.30,
    stratify=y,
    random_state=42
)

val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.50,
    stratify=y[temp_idx],
    random_state=42
)


# CREATE MASKS


train_mask = torch.zeros(num_nodes, dtype=torch.bool)
val_mask = torch.zeros(num_nodes, dtype=torch.bool)
test_mask = torch.zeros(num_nodes, dtype=torch.bool)

train_mask[train_idx] = True
val_mask[val_idx] = True
test_mask[test_idx] = True

data.train_mask = train_mask
data.val_mask = val_mask
data.test_mask = test_mask


print("\nDataset split:")
print("Train:", data.train_mask.sum().item())
print("Validation:", data.val_mask.sum().item())
print("Test:", data.test_mask.sum().item())


Dataset split:
Train: 2091
Validation: 448
Test: 449


In [ ]:
torch.save(
    data,
    "Synthetic_student_graph.pt"
)

print("\nGraph saved successfully!")


Graph saved successfully!


In [ ]:
print(data)
print("Nodes:", data.num_nodes)
print("Edges:", data.num_edges)
print("Features:", data.num_node_features)
print("Train:", data.train_mask.sum().item())
print("Validation:", data.val_mask.sum().item())
print("Test:", data.test_mask.sum().item())

Data(x=[2988, 174], edge_index=[2, 37478], y=[2988], train_mask=[2988], val_mask=[2988], test_mask=[2988])
Nodes: 2988
Edges: 37478
Features: 174
Train: 2091
Validation: 448
Test: 449
